In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib

In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

from datasets import load_dataset, Audio, DatasetDict

In [ ]:
def load_tess_speaker_independent():
    """
    Συνάρτηση υπέυθηνη για τον speaker independent διαχωρισμό. Η ομιλητής OAF
    αντισοιχίζεται στο σύνολο εκπαίδευσης και η YAF στο σύνολο δοκιμής
    """
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

In [ ]:
#Εξαγωγή των ετικετών των συναισθημάτων σε λίστα
labels = sorted(set(dataset["train"]["emotion"]))
print(labels)

#Δημιουργία λεξικών αντιστοίχισης ετικετών και αριθμών
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label

#υπολογισμός αριθμού ετικετών
num_labels = len(labels)
print(f"Διαθέσιμες ετικέτες: {labels}")
print(f"Αριθμός ετικετών: {num_labels}")

In [ ]:
# Φόρτωση του feature extractor του βασικού μοντέλου
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/data2vec-audio-base-960h")

def prepare_dataset(batch):
    audio = batch["audio"]
    #Μετατροπή δείγματος ομιλίας σε κατάλληλη μορφή για το finetuning
    batch["input_values"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    #Μεταονομασία της μεταβλητής για συμβατότητα με τον trainer
    batch["labels"] = batch["emotion"]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names)

#Εκτύπωση της δομής του νεου dataset
print("Encoded dataset features:")
print(encoded_dataset["train"].features)

#Εκτύπωση ενός δείγματος
print("\nSample from encoded dataset:")
print(encoded_dataset["train"][0])

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

def compute_metrics(eval_pred):
    """
    Συνάρτηση για τον υπολογισμό των μετρικών ανά εποχή του εποχή εκπαίδευσης και προσθέτει
    στην all_confusion_matrices. Επειδή γίνεται εκ νέου υπολογισμός της καλύτερης εποχής μετά
    το τέλος της εκπαίδευσης συτή περιέχει ένα επιπλεόν confusion matrix, ίδιο με το καλύτερο
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    num_classes = probs.shape[1]
    labels_bin = label_binarize(labels, classes=list(range(num_classes)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    roc_auc = roc_auc_score(
        labels_bin, probs, multi_class="ovr", average="macro"
    )

    # PR-AUC (macro-averaged average precision)
    pr_auc = average_precision_score(
        labels_bin, probs, average="macro"
    )

    accuracy = accuracy_score(labels, predictions)
    balanced_accuracy = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels, predictions, average="weighted")
    f1 = f1_score(labels, predictions, average="weighted")
    mcc = matthews_corrcoef(labels, predictions)
    k_score = cohen_kappa_score(labels, predictions)
    logloss = log_loss(labels, probs)

    cm = confusion_matrix(labels, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc" : mcc,
        "k_score" : k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
#Φόρτωση του αρχικού μοντελου και εισαγωγή των ετικετών
model = AutoModelForAudioClassification.from_pretrained("facebook/data2vec-audio-base-960h",
    num_labels=num_labels,
    label2id=label2id_fixed,
    id2label=id2label_fixed,
    ignore_mismatched_sizes=True
)

model.freeze_feature_encoder()

#Φόρτωση μοντέλου στην GPU
if torch.cuda.is_available():
    model = model.to('cuda')
print(f"Το μοντέλο φορτώθηκε στη συσκευή: {model.device}")

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
#Καθορισμός μετρικών εκπαίδευσης του μοντέλου
training_args = TrainingArguments(
    output_dir="./data2vec-finetuned-TESS_full",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    fp16=True,
    logging_steps=50,
    learning_rate=3e-5,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)


class DataCollatorCTCWithPadding:
    """
    Data collator για μεταβλητού μήκους δείγματα ήχου.
    Το feature_extractor.pad() αναλαμβάνει το padding των input_values ώστε όλα τα
    δείγματα ενός batch να έχουν το ίδιο μήκος. Παράλληλα, μετατρέπει τα labels από
    string σε integer ids μέσω του global λεξικού label2id, καθώς ο
    Trainer απαιτεί ακέραιες ετικέτες
    """
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        #Μετατροπή των string label σε integer
        batch["labels"] = torch.tensor([label2id[feature["labels"]] for feature in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [ ]:
#Καθαρισμός των confusion matrix, από τυχόν προηγούμενες εκτελέσεις
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
#Κλήση του trainer και εισαγωγή των παραμέτρων εκπαίδευσης
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

### Οπτικοποίηση αποτελεσμάτων (Confusion Matrices)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

### Αποθήκευση εκπαιδευμένου μοντέλου στο HuggingFace Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")